# LACRIMAE — F03 PICTOR
> *"Le Peintre rend. Frame par frame, les larmes deviennent lumière."*

**Mission** : Rendu vidéo frame par frame via Remotion (React) → `short_final.mp4`

**Prérequis** :
- `F03_PICTOR/IN/timing.json` — de F01
- `F03_PICTOR/IN/creative_config.json` — de F02
- `F03_PICTOR/IN/audio_clean.mp3` — de SHARED
- `F03_PICTOR/IN/images/` — de SHARED

---

## ÉTAPE 1 — Montage Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

## ÉTAPE 2 — Vérification des inputs (LAC_CUSTOS check-in)

In [ ]:
from pathlib import Path
import shutil, subprocess

DRIVE_BASE   = Path('/content/drive/MyDrive/DRIVE_LACRIMAE')
F03_BASE     = DRIVE_BASE / 'F03_PICTOR'
IN_DIR       = F03_BASE / 'IN'
OUT_DIR      = F03_BASE / 'OUT'
CODEBASE_DIR = F03_BASE / 'CODEBASE'
CUSTOS_PATH  = DRIVE_BASE / 'LAC_CUSTOS.py'

OUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(CUSTOS_PATH, '/content/LAC_CUSTOS.py')
result = subprocess.run(
    ['python', '/content/LAC_CUSTOS.py', '--frigate', 'F03', '--mode', 'check-in', '--drive-base', str(DRIVE_BASE)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[PICTOR] ERREUR check-in — corriger avant de continuer.')
    print(result.stderr)

## ÉTAPE 3 — Installation Node.js + Remotion

In [ ]:
# Installer Node.js 20 (LTS)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs -q
!node --version
!npm --version
print('Node.js installé.')

## ÉTAPE 4 — Préparation du projet Remotion

In [ ]:
import shutil, json
from pathlib import Path

RENDER_DIR = Path('/content/lacrimae_render')
SRC_DIR    = RENDER_DIR / 'src'
IN_RENDER  = RENDER_DIR / 'in'

# Copier tout le template src/ depuis Drive
if RENDER_DIR.exists():
    shutil.rmtree(RENDER_DIR)
shutil.copytree(CODEBASE_DIR / 'src', SRC_DIR)
shutil.copy(CODEBASE_DIR / 'src' / 'package.json', RENDER_DIR / 'package.json')

# Créer in/ avec les fichiers de données
IN_RENDER.mkdir(parents=True, exist_ok=True)
shutil.copy(IN_DIR / 'timing.json',          IN_RENDER / 'timing.json')
shutil.copy(IN_DIR / 'creative_config.json', IN_RENDER / 'creative_config.json')
shutil.copy(IN_DIR / 'audio_clean.mp3',      RENDER_DIR / 'audio_clean.mp3')

# Copier les images
img_out = RENDER_DIR / 'images'
if (IN_DIR / 'images').exists():
    shutil.copytree(IN_DIR / 'images', img_out, dirs_exist_ok=True)

# Lister les images pour les passer comme props
exts = {'.jpg', '.jpeg', '.png'}
image_files = sorted([f.name for f in img_out.iterdir() if f.suffix.lower() in exts])

print(f'Projet Remotion préparé : {RENDER_DIR}')
print(f'Images : {len(image_files)}')

## ÉTAPE 5 — Génération du Root.jsx avec les props réels

In [ ]:
import json, shutil

# Charger timing et config
with open(IN_RENDER / 'timing.json', 'r') as f:
    timing = json.load(f)
with open(IN_RENDER / 'creative_config.json', 'r') as f:
    cfg = json.load(f)

# ── PATCH PROD 2026-05-19 ──────────────────────────────────────────────────────
# Remotion ne peut pas bundler des fichiers binaires (mp3, jpg) via require()/import.
# Solution : copier les assets dans public/ et les référencer avec staticFile().
# ────────────────────────────────────────────────────────────────────────────────
PUBLIC_DIR = RENDER_DIR / 'public'
PUBLIC_IMG = PUBLIC_DIR / 'images'
PUBLIC_DIR.mkdir(exist_ok=True)
PUBLIC_IMG.mkdir(exist_ok=True)

for f in (RENDER_DIR / 'images').iterdir():
    shutil.copy(str(f), str(PUBLIC_IMG / f.name))
shutil.copy(str(RENDER_DIR / 'audio_clean.mp3'), str(PUBLIC_DIR / 'audio_clean.mp3'))

print(f'Assets copiés dans public/ : {len(image_files)} images + audio')

# Générer Root.jsx avec staticFile — pas d'imports binaires
img_array_str = '[' + ', '.join([f'"{name}"' for name in image_files]) + ']'

root_content = f'''/**
 * LACRIMAE — Root.jsx (généré automatiquement par LAC_F03.ipynb)
 * NE PAS ÉDITER MANUELLEMENT
 */
import {{ Composition, staticFile }} from "remotion";
import {{ LacrimaeShort }} from "./components/LacrimaeShort";

const timing = {json.dumps(timing, ensure_ascii=False)};
const config = {json.dumps(cfg, ensure_ascii=False)};
const imageNames = {img_array_str};
const images = imageNames.map(name => staticFile(\'images/\' + name));

export const LacrimaeRoot = () => (
  <Composition
    id="LacrimaeShort"
    component={{LacrimaeShort}}
    durationInFrames={{{timing[\'total_frames\']}}}
    fps={{{timing[\'fps\']}}}
    width={{1080}}
    height={{1920}}
    defaultProps={{{{ timing, config, images, audioSrc: staticFile(\'audio_clean.mp3\') }}}}
  />
);
'''

with open(SRC_DIR / 'Root.jsx', 'w', encoding='utf-8') as f:
    f.write(root_content)

print('Root.jsx généré (staticFile — patch prod validé 2026-05-19).')

## ÉTAPE 6 — Installation des dépendances npm

In [ ]:
import os
os.chdir(str(RENDER_DIR))
!npm install --legacy-peer-deps 2>&1 | tail -5
print('npm install terminé.')

## ÉTAPE 7 — Rendu Remotion

In [ ]:
from pathlib import Path

RENDER_OUT = Path('/content/lacrimae_render/out')
RENDER_OUT.mkdir(exist_ok=True)
FINAL_MP4  = RENDER_OUT / 'short_final.mp4'

print('[PICTOR] Lancement du rendu Remotion...')
print(f'[PICTOR] Output → {FINAL_MP4}')
print(f'[PICTOR] Frames à rendre : {timing["total_frames"]} @ {timing["fps"]}fps')

# ── PATCH PROD 2026-05-19 ──────────────────────────────────────────────────────
# Passer l'entry point explicitement (src/index.jsx) pour que Remotion v4
# resolve correctement la composition avant le path output.
# ────────────────────────────────────────────────────────────────────────────────
!npx remotion render src/index.jsx LacrimaeShort {str(FINAL_MP4)} \
  --concurrency=1 \
  --log=verbose 2>&1

## ÉTAPE 8 — Copie vers Drive + validation

In [ ]:
import shutil

if FINAL_MP4.exists():
    dest = OUT_DIR / 'short_final.mp4'
    shutil.copy(str(FINAL_MP4), str(dest))
    size_mb = dest.stat().st_size / 1_000_000
    print(f'[PICTOR] short_final.mp4 copié vers Drive → {dest} ({size_mb:.1f} Mo)')
else:
    print('[PICTOR] ERREUR — short_final.mp4 introuvable, vérifier les logs de rendu.')

## ÉTAPE 9 — Validation LAC_CUSTOS check-out

In [ ]:
!python /content/LAC_CUSTOS.py --frigate F03 --mode check-out --drive-base "{DRIVE_BASE}"

## ÉTAPE 10 — Instructions de transit

```
✓ Si LAC_CUSTOS a validé :

  Copier manuellement :
  F03_PICTOR/OUT/short_final.mp4  →  F04_SIGNUM/IN/short_final.mp4

  Puis inscrire le transit dans TRACKING/LACRIMAE_TRANSFER_LOG.md
```

### Reprise après interruption Colab (CHECKPOINT SACRÉ)

Si Colab déconnecte pendant le rendu :
1. Remonter Drive
2. Relancer depuis ÉTAPE 3
3. Remotion reprend là où il s'est arrêté grâce au cache de frames